# 02 — Fine-tune Gemma 2B with QLoRA
**AI Travel Assistant Chatbot — SRH Applied AI Project**

This notebook fine-tunes **Google Gemma 2B-IT** on the travel SFT dataset prepared in notebook 01.

**Technique:** QLoRA (4-bit quantization + Low-Rank Adaptation)
- Runs on a **free Google Colab T4 GPU** (16 GB VRAM)
- Training time: ~25-40 minutes on T4

**Prerequisites:**
1. Run `01_dataset_preparation.ipynb` first to generate `data/travel_sft_train.jsonl`
2. Have a HuggingFace account + access token (free)
3. Request access to `google/gemma-2-2b-it` on HuggingFace (usually approved instantly)

**Output:** LoRA adapter saved to `checkpoints/gemma-travel-lora/`

## Step 0 — Verify GPU

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else "No GPU found — switch to GPU runtime in Colab!")

## Step 1 — Install dependencies

> This may take 2-3 minutes on Colab. Restart the runtime after installation.

In [ ]:
!pip install -q \
    transformers==4.45.2 \
    peft==0.13.2 \
    trl==0.11.3 \
    bitsandbytes==0.43.3 \
    accelerate==0.34.2 \
    datasets==3.0.1 \
    huggingface_hub

print("Installation complete!")

## Step 2 — HuggingFace login

You need a HuggingFace token to download the Gemma model:
1. Go to https://huggingface.co/settings/tokens
2. Create a token with **Read** permission
3. Accept Gemma's license at https://huggingface.co/google/gemma-2-2b-it
4. Paste the token below

In [ ]:
from huggingface_hub import login

# Option A: interactive login (recommended for Colab)
login()  # This opens a dialog to paste your token

# Option B: paste token directly (less secure)
# login(token="hf_your_token_here")

## Step 3 — Imports and configuration

In [ ]:
import os
import json
import pathlib
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from trl import SFTTrainer, SFTConfig

# ─── Configuration ───────────────────────────────────────────────────────────
BASE_MODEL   = "google/gemma-2-2b-it"         # The base Gemma model
DATA_PATH    = "data/travel_sft_train.jsonl"   # From notebook 01
OUTPUT_DIR   = "checkpoints/gemma-travel-lora" # Where adapter is saved

# LoRA hyperparameters
LORA_R       = 16     # Rank — higher = more capacity, more memory
LORA_ALPHA   = 32     # Scaling factor (usually 2 * r)
LORA_DROPOUT = 0.05

# Training hyperparameters
EPOCHS       = 3
BATCH_SIZE   = 2      # Per-device batch size
GRAD_ACCUM   = 4      # Effective batch = 2 * 4 = 8
LR           = 2e-4
MAX_SEQ_LEN  = 1024

print(f"Base model : {BASE_MODEL}")
print(f"Data path  : {DATA_PATH}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None — using CPU (slow!)'}") 

## Step 4 — Load and inspect training data

In [ ]:
if not pathlib.Path(DATA_PATH).exists():
    raise FileNotFoundError(
        f"Training data not found at '{DATA_PATH}'.\n"
        "Please run notebook 01_dataset_preparation.ipynb first!"
    )

raw_ds = load_dataset("json", data_files=DATA_PATH, split="train")
print(f"Training examples: {len(raw_ds)}")
print("\nFirst example messages:")
for msg in raw_ds[0]["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

## Step 5 — Load tokenizer and format dataset

In [ ]:
print(f"Loading tokenizer for {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.padding_side = "right"  # Important for Gemma

def format_with_chat_template(example: dict) -> dict:
    """Apply Gemma chat template to messages list."""
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

formatted_ds = raw_ds.map(format_with_chat_template, remove_columns=raw_ds.column_names)

print("\nFormatted example:")
print(formatted_ds[0]["text"][:500], "...")

## Step 6 — Load Gemma in 4-bit (QLoRA)

4-bit quantization reduces GPU memory from ~16 GB → ~5 GB, making this fit on a free T4.

In [ ]:
print("Configuring 4-bit quantization (QLoRA)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,  # Extra compression
    bnb_4bit_quant_type="nf4",       # NormalFloat4 — best for LLM weights
    bnb_4bit_compute_dtype=torch.bfloat16,
)

print(f"Loading {BASE_MODEL} in 4-bit...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="eager",  # Avoid Flash Attention dependency
)

# Prepare for k-bit training (freezes quantized weights, enables gradient checkpointing)
model = prepare_model_for_kbit_training(model)

print("Model loaded successfully!")
print(f"Model dtype: {model.dtype}")

## Step 7 — Apply LoRA adapter

LoRA adds small trainable matrices (rank `r`) to the attention and MLP layers.
Only ~0.5% of parameters are trained — making it fast and memory-efficient.

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    # Target all projection layers in attention + feed-forward
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)

# Show how many parameters are actually being trained
model.print_trainable_parameters()

# Expected output: "trainable params: ~27M || all params: ~2.5B || trainable%: ~1.0%"

## Step 8 — Configure trainer and start fine-tuning

In [ ]:
pathlib.Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR,
    bf16=True,                          # Use bfloat16 on T4
    logging_steps=10,
    save_strategy="epoch",
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    report_to="none",                   # Disable wandb/tensorboard
    gradient_checkpointing=True,        # Save VRAM at cost of slightly slower training
    optim="paged_adamw_8bit",           # 8-bit optimizer — saves VRAM
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted_ds,
    tokenizer=tokenizer,
)

print("Starting fine-tuning...")
print(f"Epochs: {EPOCHS} | Batch size: {BATCH_SIZE} | Grad accum: {GRAD_ACCUM}")
print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print("="*50)

trainer.train()

## Step 9 — Save the LoRA adapter

In [ ]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"\nLoRA adapter saved to: {OUTPUT_DIR}/")
print("\nFiles saved:")
for f in pathlib.Path(OUTPUT_DIR).iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name:<40} {size_mb:.1f} MB")

print("\n" + "="*50)
print("NEXT STEPS:")
print(f"1. Set LORA_ADAPTER_PATH={OUTPUT_DIR} in backend/.env")
print("2. Run notebook 03_evaluation.ipynb to measure performance")
print("="*50)

## Step 10 — Quick inference test

Let's verify the fine-tuned model can answer travel questions correctly.

In [ ]:
from peft import PeftModel
from transformers import pipeline

TEST_PROMPTS = [
    "Plan a 3-day trip to Paris for a couple interested in art and food.",
    "What is the best time to visit Thailand?",
    "What documents do I need for a Schengen visa?",
]

def generate_answer(prompt: str, max_new_tokens: int = 300) -> str:
    messages = [{"role": "user", "content": prompt}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

print("=" * 60)
print("FINE-TUNED MODEL — INFERENCE TEST")
print("=" * 60)
for i, prompt in enumerate(TEST_PROMPTS, 1):
    print(f"\n[Test {i}] Question: {prompt}")
    print("-" * 40)
    answer = generate_answer(prompt)
    print(f"Answer: {answer[:500]}..." if len(answer) > 500 else f"Answer: {answer}")
    print()

## Done!

Fine-tuning complete. Your LoRA adapter is saved at `checkpoints/gemma-travel-lora/`.

**Next step:** Open `03_evaluation.ipynb` to run the formal evaluation and prove ~80%+ similarity score.